## Инициализация

### Импорты

In [ ]:
import numpy as np
from numpy import pi, sin, cos
from numpy.typing import NDArray

from matplotlib import pyplot as plt
plt.rcParams["figure.dpi"] = 300

from typing import Callable

from time import perf_counter

from mcsolve import (
    Solver,
    Poisson,
    BoundarySegment, Boundary, Domain,
    Line, Arc,
    Dirichlet, Neumann
)

### Утилиты

In [ ]:
def visualize(solver: Solver):
    fig, axs = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    axs[1].axis("off")
    ax = axs[0]

    ax.grid(True)
    ax.set_aspect("equal", adjustable="box")
    assembler = solver.assembler
    domain = assembler.domain
    h = assembler.h

    ax.set_xlim([domain.xmin - h, domain.xmax + h])
    ax.set_ylim([domain.ymin - h, domain.ymax + h])

    inner_points = np.array(assembler.inner.points)
    boundary_points = np.array(assembler.boundary.points)
    ghost_points = np.array(assembler.ghost.points)
    mirror_points = np.array(assembler.mirror.points)

    ax.scatter(
        inner_points[:, 0],
        inner_points[:, 1],
        s=5, c="black")
    
    if boundary_points.size:
        ax.scatter(
            boundary_points[:, 0], 
            boundary_points[:, 1], 
            s=5, c="green")
    
    if ghost_points.size:
        ax.scatter(
            ghost_points[:, 0],
            ghost_points[:, 1],
            s=5, c="red")
        
        ax.scatter(
            mirror_points[:, 0],
            mirror_points[:, 1],
            s=5, c="blue")

    ax.set_title("Визуализация дискретизации")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    
    plt.show()

def relative_error(exact_value: NDArray[np.float64] | float,
                   approximation: NDArray[np.float64] | float) -> float:
    return np.linalg.norm(approximation - exact_value) / np.linalg.norm(exact_value)

def compare(solver: Solver, number_of_walks: int, max_steps: int, u: Callable):
    approximation = solver.solve(max_steps, number_of_walks)
    
    assembler = solver.assembler
    nx, ny = assembler.nx, assembler.ny
    domain = assembler.domain
    
    h = assembler.h

    xmin, xmax = domain.xmin - h, domain.xmax + h
    ymin, ymax = domain.ymin - h, domain.ymax + h

    x, y = np.meshgrid(
        np.linspace(xmin, xmax, nx), 
        np.linspace(ymin, ymax, ny)
    )

    exact_values = np.full_like(x, np.nan)
    for j in range(ny):
        for i in range(nx):
            if not np.isnan(approximation[j, i]):
                exact_values[j,i] = u(x[j, i], y[j, i])

    mask = ~np.isnan(approximation)

    approximate_values = approximation[mask]
    exact_values = exact_values[mask]

    exact_grid = np.full_like(approximation, np.nan)
    exact_grid[mask] = exact_values

    print(f"Относительная ошибка: {relative_error(exact_values, approximate_values):.3e}")

    fig, axs = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    
    ax1, ax2 = axs

    ax1.grid(True)
    img1 = ax1.imshow(
        exact_grid, origin="lower", extent=(xmin, xmax, ymin, ymax),
        interpolation="nearest", cmap="plasma")
    ax1.set_title("Точное решение"); ax1.set_xlabel("x"); ax1.set_ylabel("y")
    fig.colorbar(img1, ax=ax1)

    ax2.grid(True)
    img2 = ax2.imshow(
        approximation, origin="lower", extent=(xmin, xmax, ymin, ymax),
        interpolation="nearest", cmap="plasma")
    ax2.set_title("Приблизительное решение"); ax2.set_xlabel("x"); ax2.set_ylabel("y")
    fig.colorbar(img2, ax=ax2)

    plt.show()


### Уравнение и граничные условия

In [ ]:
u = lambda x, y: sin(pi*x)*sin(pi*y) # точное значение

f = lambda x, y: -2.0*pi**2*u(x, y) # правая часть

equation = Poisson(f)

dirichlet_condition = Dirichlet(u)

gradient = lambda x, y: np.array([pi*cos(pi*x)*sin(pi*y), pi*sin(pi*x)*cos(pi*y)])
neumann_condition = lambda shape: Neumann(lambda x, y: np.dot(gradient(x, y), shape.normal(np.array([x, y]))))

### Гиперпараметры

In [ ]:
max_steps = 8000

number_of_walks = 8000

h = 0.02

## Область с острым углом

In [ ]:
p0 = np.array([0.10, 0.10])
p1 = np.array([0.50, 0.90])
p2 = np.array([0.90, 0.10])

line0 = Line(p0, p1)
line1 = Line(p1, p2)
line2 = Line(p2, p0)

triangle = Boundary([
    BoundarySegment(line0, neumann_condition(line0)),
    BoundarySegment(line1, neumann_condition(line1)),
    BoundarySegment(line2, dirichlet_condition),
])

domain = Domain(
    0.0, 1.0,
    0.0, 1.0,
    [triangle]
)

solver = Solver(domain, equation, h)

# solver.solve(max_steps, number_of_walks)
# solver.solve_at(x, y, max_steps, number_of_walks)

visualize(solver)

compare(solver, number_of_walks, max_steps, u)

## Многосвязная область

In [ ]:
c0 = np.array([0.50, 0.50])
p0 = np.array([0.50, 0.95])

outer_circle = Boundary(
    [BoundarySegment(Arc(c0, p0, p0), dirichlet_condition)]
)

c1 = np.array([0.50, 0.50])
p1 = np.array([0.50, 0.75])

inner_circle = Boundary(
    [ BoundarySegment(Arc(c1, p1, p1, False), dirichlet_condition) ]
)

domain = Domain(
    0.0, 1.0,
    0.0, 1.0,
    [outer_circle, inner_circle]    
)

solver = Solver(domain, equation, h)

visualize(solver)

compare(solver, number_of_walks, max_steps, u)

## Область с вогнутым углом

In [ ]:
c0 = np.array([0.50, 0.50])
p0 = np.array([0.90, 0.29])
p1 = np.array([0.90, 0.71])

arc = Arc(c0, p0, p1)
line0 = Line(p1, c0)
line1 = Line(c0, p0)

pie = Boundary([
    BoundarySegment(arc, dirichlet_condition),
    BoundarySegment(line0, neumann_condition(line0)),
    BoundarySegment(line1, neumann_condition(line1)),
])

domain = Domain(
    0.0, 1.0,
    0.0, 1.0,
    [pie]   
)

solver = Solver(domain, equation, h)

visualize(solver)

compare(solver, number_of_walks, max_steps, u)

## Область с узкой перемычкой

In [ ]:
d = 0.1

points = [
    np.array([0.05, 0.05]), np.array([0.05, 0.95]),
    np.array([0.30, 0.95]),
    np.array([0.30, 0.5 + 0.5*d]),
    np.array([0.70, 0.5 + 0.5*d]),
    np.array([0.70, 0.95]), np.array([0.95, 0.95]),
    np.array([0.95, 0.05]), np.array([0.70, 0.05]),
    np.array([0.70, 0.5 - 0.5*d]),
    np.array([0.30, 0.5 - 0.5*d]),
    np.array([0.30, 0.05]),
]
points.append(points[0])

segments = [BoundarySegment(Line(points[i], points[i+1]), dirichlet_condition) 
            for i in range(len(points)-1)]

dumbbell = Boundary(segments)

domain = Domain(
    0.0, 1.0, 
    0.0, 1.0,
    boundaries=[dumbbell]
)

solver = Solver(domain, equation, h)

visualize(solver)

compare(solver, number_of_walks, max_steps, u)


## Локальное вычисление

In [ ]:
number_of_solves = 1000

x = 0.23579
y = 0.4321

start = perf_counter()
for _ in range(number_of_solves):
    solver.solve_at(x, y, max_steps, number_of_walks)
end = perf_counter()

print(f"Среднее время расчёта: {(end - start) / number_of_solves*1000:.3f} мс")